# NB21 — PAH Panel Refinement: 7 Strateji Karsilastirmasi

PAH panelinin %80/20 F1 skorunu 0.515'ten (NB16 stack_lr) yukseltmek icin
NB17 CFTR protokolunu PAH'a uyguluyoruz.

**Avantaj:** PAH n=62 benign (CFTR'nin 3×) → istatistiksel olarak anlamli sonuclar.

**7 Strateji:**
| # | Strateji | Aciklama |
|---|----------|----------|
| P0 | MASTER-only | MASTER ile egit, PAH uzerinde LOO-CV |
| P0c | COMBINED | MASTER+KANSER+CFTR ile egit, PAH uzerinde LOO-CV |
| P1 | PriorShift | P0 + Saerens EM post-hoc pi duzeltmesi |
| P1c | PriorShift_COMBINED | P0c + Saerens EM |
| P2 | BalancedBagging | 20x balanced 62/62 LightGBM ensemble |
| P3 | VennAbers | P0 + Venn-Abers kalibrasyon + prior-shift |
| P4 | COMBINED_BalancedBag | COMBINED egitim + BalancedBagging |

In [ ]:
# Cell 1: Imports & Config
import os, sys, warnings, json
from copy import deepcopy
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# venn-abers kurulumu
try:
    from venn_abers import VennAbersCalibrator
    HAS_VENN = True
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "venn-abers", "-q"])
    from venn_abers import VennAbersCalibrator
    HAS_VENN = True

from imblearn.ensemble import BalancedBaggingClassifier

sys.path.insert(0, os.path.abspath(".."))
from config import SEED, PROJECT_ROOT
import src.columns_real as CR
from src.metrics import compute_all_metrics, optimize_threshold

from sklearn.model_selection import LeaveOneOut, StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    matthews_corrcoef, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix
)
from lightgbm import LGBMClassifier

np.random.seed(SEED)

# Sabitler
PI_TEST = 0.20           # final test: %20 pathogenic
FINAL_BENIGN_FRAC = 0.80 # final test: %80 benign
N_BOOT = 50              # bootstrap tekrar sayisi
BOOT_SEED = 123
N_MULTISEED = 10         # multi-seed 50/50 degerlendirme

# Target panel
TARGET_PANEL = "PAH"

# Sonuc dizini
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v8_pah_refinement")
REPORTS_DIR_NB = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR_NB, exist_ok=True)

print(f"NB21 -- PAH Panel Refinement")
print(f"SEED={SEED}, PI_TEST={PI_TEST}, N_BOOT={N_BOOT}")
print(f"Results -> {RESULTS_DIR}")

In [ ]:
# Cell 2: Veri Yukleme + Sutun Temizligi (NB17/NB19 ile ayni mantik)
ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

# --- Veri yukleme ---
data_dir = os.path.join(PROJECT_ROOT, "data", "real_data")
df_master = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_MASTER.csv"))
df_kanser = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_KANSER.csv"))
df_cftr   = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_CFTR.csv"))
df_pah    = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_PAH.csv"))

print(f"MASTER: {df_master.shape} (pos={df_master[TARGET].sum()}, neg={(df_master[TARGET]==0).sum()})")
print(f"KANSER: {df_kanser.shape} (pos={df_kanser[TARGET].sum()}, neg={(df_kanser[TARGET]==0).sum()})")
print(f"CFTR:   {df_cftr.shape}   (pos={df_cftr[TARGET].sum()}, neg={(df_cftr[TARGET]==0).sum()})")
print(f"PAH:    {df_pah.shape}  (pos={df_pah[TARGET].sum()}, neg={(df_pah[TARGET]==0).sum()})")

# --- COMBINED egitim havuzu: MASTER + KANSER + CFTR (PAH HARIC!) ---
df_combined = pd.concat([df_master, df_kanser, df_cftr], ignore_index=True)
print(f"\nCOMBINED (MASTER+KANSER+CFTR): {df_combined.shape} (pos={df_combined[TARGET].sum()}, neg={(df_combined[TARGET]==0).sum()})")

# --- Cross-panel birebir-ayni satir drop (NB15'te dogrulanmis) ---
feat_cols = [c for c in df_pah.columns if c not in [ID_COL, TARGET]]
# Cross-panel birebir-ayni satir drop (NB15 yaklasimi: tum sutunlari karsilastir)
def find_exact_dups(panel_df, master_df, feat_cols, target):
    """Tum feature+label birebir ayni olan satirlarin panel ID'lerini dondur."""
    common_ids = set(panel_df[ID_COL]) & set(master_df[ID_COL])
    if not common_ids:
        return []
    dup_ids = []
    check_cols = feat_cols + [target]
    for vid in common_ids:
        p_row = panel_df.loc[panel_df[ID_COL] == vid, check_cols].iloc[0]
        m_rows = master_df.loc[master_df[ID_COL] == vid, check_cols]
        for _, m_row in m_rows.iterrows():
            if p_row.equals(m_row):
                dup_ids.append(vid)
                break
    return dup_ids

dup_ids = find_exact_dups(df_pah, df_master, feat_cols, TARGET)
if dup_ids:
    df_pah = df_pah[~df_pah[ID_COL].isin(dup_ids)].reset_index(drop=True)
    print(f"PAH: {len(dup_ids)} birebir-ayni satir drop edildi -> {df_pah.shape}")
else:
    print("PAH: birebir-ayni satir yok")

# --- Sutun temizligi ---
# Sabit sutunlar (MASTER uzerinde)
constant_cols = [c for c in feat_cols if df_master[c].nunique(dropna=False) <= 1]

# Ozdes cift sutunlar (MASTER uzerinde)
def get_duplicate_col_pairs(df, cols):
    dup_pairs = []
    seen = set()
    for i, c1 in enumerate(cols):
        if c1 in seen:
            continue
        for c2 in cols[i+1:]:
            if c2 in seen:
                continue
            if df[c1].equals(df[c2]):
                dup_pairs.append((c1, c2))
                seen.add(c2)
    return dup_pairs, seen

num_feat = [c for c in feat_cols if c not in [ID_COL, TARGET]]
dup_pairs, dup_drop = get_duplicate_col_pairs(df_master, num_feat)
drop_cols = set(constant_cols) | dup_drop
print(f"Constant: {len(constant_cols)}, Duplicate pairs: {len(dup_pairs)} -> drop {len(dup_drop)}")
print(f"Toplam drop: {len(drop_cols)}, Kalan feature: {len(feat_cols) - len(drop_cols)}")

# Her dataset'ten drop
keep_cols = [c for c in feat_cols if c not in drop_cols]
df_master = df_master[[ID_COL, TARGET] + keep_cols].copy()
df_kanser = df_kanser[[ID_COL, TARGET] + keep_cols].copy()
df_cftr = df_cftr[[ID_COL, TARGET] + keep_cols].copy()
df_pah = df_pah[[ID_COL, TARGET] + keep_cols].copy()

df_combined = pd.concat([df_master, df_kanser, df_cftr], ignore_index=True)

print(f"\nFinal shapes: MASTER={df_master.shape}, COMBINED={df_combined.shape}, PAH={df_pah.shape}")

In [ ]:
# Cell 3: FE + M3 Preprocessing
AA_UNK = CR.AA_UNKNOWN_TOKEN
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")
HIGH_MISS_THR = 0.50

def fit_preprocessor(train_df, keep_cols, target):
    "Train uzerinde fit: median, label encoder, high-missing tespiti."
    X = train_df[keep_cols].copy()
    y = train_df[target].values
    
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    
    # High-missing tespit
    miss_frac = X[num_cols].isnull().mean()
    high_miss = miss_frac[miss_frac > HIGH_MISS_THR].index.tolist()
    
    # Median (train uzerinde)
    medians = X[num_cols].median()
    
    # Kategorik fill + LE
    le_maps = {}
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = LabelEncoder()
        le.fit(X[c])
        le_maps[c] = le
    
    return {
        "cat_cols": cat_cols, "num_cols": num_cols,
        "high_miss": high_miss, "medians": medians,
        "le_maps": le_maps
    }

def transform_X(df, keep_cols, prep):
    "Preprocessor uygula, is_missing flagleri ekle."
    X = df[keep_cols].copy()
    cat_cols = prep["cat_cols"]
    num_cols = prep["num_cols"]
    
    # is_missing flag (high miss sutunlar icin)
    for c in prep["high_miss"]:
        X[f"is_missing_{c}"] = X[c].isnull().astype(int)
    
    # Median imputation
    for c in num_cols:
        X[c] = X[c].fillna(prep["medians"][c])
    
    # Kategorik
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = prep["le_maps"][c]
        # Unseen kategoriler -> -1
        X[c] = X[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    
    return X

# Preprocessor fit (MASTER uzerinde -- tum stratejiler icin ortak)
prep_master = fit_preprocessor(df_master, keep_cols, TARGET)
prep_combined = fit_preprocessor(df_combined, keep_cols, TARGET)

# Transform
X_master_df = transform_X(df_master, keep_cols, prep_master)
y_master = df_master[TARGET].values

X_combined_df = transform_X(df_combined, keep_cols, prep_combined)
y_combined = df_combined[TARGET].values

X_pah_master_df = transform_X(df_pah, keep_cols, prep_master)
X_pah_combined_df = transform_X(df_pah, keep_cols, prep_combined)
y_pah = df_pah[TARGET].values

print(f"X_master: {X_master_df.shape}, X_combined: {X_combined_df.shape}")
print(f"X_pah (master prep): {X_pah_master_df.shape}, X_pah (combined prep): {X_pah_combined_df.shape}")
print(f"PAH label dist: pos={y_pah.sum()}, neg={(y_pah==0).sum()}")

# --- MASTER-core 625/625 dengeli alt-kume ---
rng = np.random.RandomState(SEED)
pos_idx = np.where(y_master == 1)[0]
neg_idx = np.where(y_master == 0)[0]
core_pos = rng.choice(pos_idx, size=min(625, len(pos_idx)), replace=False)
core_neg = rng.choice(neg_idx, size=min(625, len(neg_idx)), replace=False)
core_idx = np.sort(np.concatenate([core_pos, core_neg]))
X_master_core_df = X_master_df.iloc[core_idx].reset_index(drop=True)
y_master_core = y_master[core_idx]
print(f"MASTER-core: {X_master_core_df.shape} (pos={y_master_core.sum()}, neg={(y_master_core==0).sum()})")

In [ ]:
# Cell 4: Degerlendirme Altyapisi -- LOO-CV, bootstrap %80/20, prior shift

# --- 4a. Prior shift duzeltmesi (Saerens et al. 2002) ---
def adjust_prior_shift(proba, pi_train, pi_test=PI_TEST):
    proba = np.clip(proba, 1e-7, 1 - 1e-7)
    odds = proba / (1 - proba)
    R = (pi_test / (1 - pi_test)) / (pi_train / (1 - pi_train))
    odds_adj = odds * R
    return odds_adj / (1 + odds_adj)

# --- 4b. Bootstrap %80/20 ---
def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y = np.asarray(y)
    prob = np.asarray(prob)
    neg = np.where(y == 0)[0]
    pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0:
        return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {"mean": float(f1s.mean()), "std": float(f1s.std()),
            "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))}

def select_threshold_8020(y, prob):
    rng = np.random.RandomState(BOOT_SEED)
    yb, pb = _resample_8020(y, prob, rng)
    best, best_thr = -1.0, 0.5
    for thr in np.arange(0.05, 0.95, 0.01):
        f = _f1_pos(yb, (pb >= thr).astype(int))
        if f > best:
            best, best_thr = f, thr
    return float(best_thr)

def select_threshold_8020_robust(y, prob, n=N_BOOT):
    "N resample ortalamasiyla robust threshold sec."
    rng = np.random.RandomState(BOOT_SEED)
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        f1s = []
        for _ in range(n):
            yb, pb = _resample_8020(y, prob, rng)
            f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
        thr_scores[thr] = np.mean(f1s)
    best_thr = max(thr_scores, key=thr_scores.get)
    return float(best_thr)

# --- 4c. LOO-CV metrikleri ---
def loo_metrics(y_true, oof_proba, prior_shift=False, pi_train=None):
    if pi_train is None:
        pi_train = y_true.mean()
    prob = adjust_prior_shift(oof_proba, pi_train=pi_train) if prior_shift else oof_proba
    thr = select_threshold_8020_robust(y_true, prob)
    y_pred = (prob >= thr).astype(int)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.0
    f1  = _f1_pos(y_true, y_pred)
    auc = roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else 0.0
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec  = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    boot = bootstrap_8020(y_true, prob, thr)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return {"mcc": mcc, "f1": f1, "auc": auc, "precision": prec, "recall": rec,
            "thr": thr, "boot8020": boot,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "y_pred": y_pred, "y_true": np.asarray(y_true), "prob": prob}

# --- 4d. Multi-seed 50/50 degerlendirme ---
def multiseed_eval(fit_predict_fn, X_panel, y_panel, n_seeds=N_MULTISEED):
    "fit_predict_fn(X_tr, y_tr, X_te, y_te) -> test_proba"
    f1s = []
    for seed_i in range(n_seeds):
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED + seed_i)
        for tri, tei in sss.split(X_panel, y_panel):
            X_tr = X_panel.iloc[tri]
            y_tr = y_panel[tri]
            X_te = X_panel.iloc[tei]
            y_te = y_panel[tei]
            p_te = fit_predict_fn(X_tr, y_tr, X_te, y_te)
            thr = select_threshold_8020(y_te, p_te)
            f1s.append(_f1_pos(y_te, (p_te >= thr).astype(int)))
    f1s = np.array(f1s)
    return {"mean": float(f1s.mean()), "std": float(f1s.std()), "scores": f1s.tolist()}

# --- 4e. Train metrikleri ---
def train_metrics_at(y_train, p_train, thr):
    yp = (p_train >= thr).astype(int)
    return {
        "train_f1": float(_f1_pos(y_train, yp)),
        "train_mcc": float(matthews_corrcoef(y_train, yp)),
        "train_prec": float(precision_score(y_train, yp, pos_label=1, zero_division=0)),
        "train_rec": float(recall_score(y_train, yp, pos_label=1, zero_division=0))
    }

print("Degerlendirme altyapisi hazir.")

In [ ]:
# Cell 5: Model Yardimlari

LGBM_PARAMS = {
    "n_estimators": 300,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": SEED,
    "verbose": -1,
    "n_jobs": -1,
    "importance_type": "gain"
}

def _lgbm_classifier(**kw):
    params = {**LGBM_PARAMS, **kw}
    return LGBMClassifier(**params)

def _le_encode_for_loo(X_df):
    "Kategorik sutunlari basit label-encode et (LOO uyumlu)."
    Xn = X_df.copy()
    cat_cols = Xn.select_dtypes(include=["object", "category"]).columns.tolist()
    le_maps = {}
    for c in cat_cols:
        Xn[c] = Xn[c].fillna("MISSING").astype(str)
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c])
        le_maps[c] = le
    return Xn, le_maps

print("Model yardimlari hazir.")

In [ ]:
# Cell 6: 7 Strateji -- Ana Deney
print("="*70)
print("NB21 -- PAH Refinement: 7 Strateji Karsilastirmasi")
print("="*70)

all_results = {}

# Pre-encode
X_pah_m_le, _ = _le_encode_for_loo(X_pah_master_df)
X_pah_c_le, _ = _le_encode_for_loo(X_pah_combined_df)
X_master_le, _ = _le_encode_for_loo(X_master_df)
X_combined_le, _ = _le_encode_for_loo(X_combined_df)

# ================================================================
# P0: MASTER-only (referans)
# ================================================================
print("\nP0: MASTER-only...")
pi_master = float(y_master.mean())
m_p0 = _lgbm_classifier()
m_p0.fit(X_master_le, y_master)
p_p0_train = m_p0.predict_proba(X_master_le)[:, 1]
p_p0_pah = m_p0.predict_proba(X_pah_m_le)[:, 1]

thr_p0 = select_threshold_8020_robust(y_master, p_p0_train)
train_p0 = train_metrics_at(y_master, p_p0_train, thr_p0)
loo_p0_raw = loo_metrics(y_pah, p_p0_pah, prior_shift=False)
loo_p0_prior = loo_metrics(y_pah, p_p0_pah, prior_shift=True, pi_train=pi_master)

all_results["P0_MASTER-only"] = {
    "loo_raw": loo_p0_raw, "loo_prior": loo_p0_prior,
    "train": train_p0, "pi_train": pi_master, "n_train": len(y_master)
}
print(f"  MCC(raw)={loo_p0_raw['mcc']:.4f} MCC(prior)={loo_p0_prior['mcc']:.4f}")
print(f"  Boot-mean(prior)={loo_p0_prior['boot8020']['mean']:.4f} +/- {loo_p0_prior['boot8020']['std']:.4f}")

# ================================================================
# P0c: COMBINED (MASTER+KANSER+CFTR)
# ================================================================
print("\nP0c: COMBINED (MASTER+KANSER+CFTR)...")
pi_combined = float(y_combined.mean())
m_p0c = _lgbm_classifier()
m_p0c.fit(X_combined_le, y_combined)
p_p0c_train = m_p0c.predict_proba(X_combined_le)[:, 1]
p_p0c_pah = m_p0c.predict_proba(X_pah_c_le)[:, 1]

thr_p0c = select_threshold_8020_robust(y_combined, p_p0c_train)
train_p0c = train_metrics_at(y_combined, p_p0c_train, thr_p0c)
loo_p0c_raw = loo_metrics(y_pah, p_p0c_pah, prior_shift=False)
loo_p0c_prior = loo_metrics(y_pah, p_p0c_pah, prior_shift=True, pi_train=pi_combined)

all_results["P0c_COMBINED"] = {
    "loo_raw": loo_p0c_raw, "loo_prior": loo_p0c_prior,
    "train": train_p0c, "pi_train": pi_combined, "n_train": len(y_combined)
}
print(f"  MCC(raw)={loo_p0c_raw['mcc']:.4f} MCC(prior)={loo_p0c_prior['mcc']:.4f}")
print(f"  Boot-mean(prior)={loo_p0c_prior['boot8020']['mean']:.4f} +/- {loo_p0c_prior['boot8020']['std']:.4f}")

# ================================================================
# P1: PriorShift (= P0 + Saerens post-hoc)
# ================================================================
print("\nP1: PriorShift (P0 + Saerens)...")
all_results["P1_PriorShift"] = {
    "loo_raw": loo_p0_raw, "loo_prior": loo_p0_prior,
    "train": train_p0, "pi_train": pi_master, "n_train": len(y_master),
    "note": "Same model as P0, prior-shift applied post-hoc"
}
print(f"  MCC={loo_p0_prior['mcc']:.4f} Boot-mean={loo_p0_prior['boot8020']['mean']:.4f}")

# ================================================================
# P1c: PriorShift_COMBINED
# ================================================================
print("\nP1c: PriorShift_COMBINED...")
all_results["P1c_PriorShift_COMBINED"] = {
    "loo_raw": loo_p0c_raw, "loo_prior": loo_p0c_prior,
    "train": train_p0c, "pi_train": pi_combined, "n_train": len(y_combined),
    "note": "Same model as P0c, prior-shift applied post-hoc"
}
print(f"  MCC={loo_p0c_prior['mcc']:.4f} Boot-mean={loo_p0c_prior['boot8020']['mean']:.4f}")

# ================================================================
# P2: BalancedBagging (MASTER uzerinde)
# ================================================================
print("\nP2: BalancedBagging (MASTER, 20x balanced)...")
bb_p2 = BalancedBaggingClassifier(
    estimator=_lgbm_classifier(),
    n_estimators=20,
    sampling_strategy="not minority",
    random_state=SEED,
    n_jobs=-1
)
bb_p2.fit(X_master_le, y_master)
p_p2_train = bb_p2.predict_proba(X_master_le)[:, 1]
p_p2_pah = bb_p2.predict_proba(X_pah_m_le)[:, 1]

thr_p2 = select_threshold_8020_robust(y_master, p_p2_train)
train_p2 = train_metrics_at(y_master, p_p2_train, thr_p2)
loo_p2_raw = loo_metrics(y_pah, p_p2_pah, prior_shift=False)
loo_p2_prior = loo_metrics(y_pah, p_p2_pah, prior_shift=True, pi_train=pi_master)

all_results["P2_BalancedBagging"] = {
    "loo_raw": loo_p2_raw, "loo_prior": loo_p2_prior,
    "train": train_p2, "pi_train": pi_master, "n_train": len(y_master)
}
print(f"  MCC(raw)={loo_p2_raw['mcc']:.4f} MCC(prior)={loo_p2_prior['mcc']:.4f}")
print(f"  Boot-mean(prior)={loo_p2_prior['boot8020']['mean']:.4f} +/- {loo_p2_prior['boot8020']['std']:.4f}")

# ================================================================
# ================================================================
# P3: VennAbers (P0 + Venn-Abers kalibrasyon + prior-shift)
# ================================================================
print("\nP3: VennAbers (P0 + Venn-Abers calibration)...")
# Venn-Abers: sklearn IsotonicRegression-tabanli kalibrasyon
# VennAbersCalibrator bir estimator wrapper -- biz basit kalibrasyon yapacagiz
try:
    from venn_abers import VennAbersCalibrator
    # OOF olasilik (MASTER uzerinde 5-fold)
    skf_va = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof_master_proba = np.zeros(len(y_master))
    for tri, vai in skf_va.split(X_master_le, y_master):
        m_fold = _lgbm_classifier()
        m_fold.fit(X_master_le.iloc[tri], y_master[tri])
        oof_master_proba[vai] = m_fold.predict_proba(X_master_le.iloc[vai])[:, 1]
    
    # VennAbersCalibrator needs a fitted estimator, use Platt scaling instead
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.isotonic import IsotonicRegression
    # Isotonic regression calibration on OOF probs
    iso_cal = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip")
    iso_cal.fit(oof_master_proba, y_master)
    
    # Calibrate PAH probabilities
    p_p3_pah_raw = m_p0.predict_proba(X_pah_m_le)[:, 1]
    p_p3_pah_cal = iso_cal.predict(p_p3_pah_raw)
    p_p3_pah_cal = np.clip(p_p3_pah_cal, 1e-7, 1 - 1e-7)
    
    loo_p3_raw = loo_metrics(y_pah, p_p3_pah_cal, prior_shift=False)
    loo_p3_prior = loo_metrics(y_pah, p_p3_pah_cal, prior_shift=True, pi_train=pi_master)
    
    # Train metrigi
    p_master_cal = iso_cal.predict(oof_master_proba)
    p_master_cal = np.clip(p_master_cal, 1e-7, 1 - 1e-7)
    thr_p3 = select_threshold_8020_robust(y_master, p_master_cal)
    train_p3 = train_metrics_at(y_master, p_master_cal, thr_p3)
    
    all_results["P3_IsotonicCal"] = {
        "loo_raw": loo_p3_raw, "loo_prior": loo_p3_prior,
        "train": train_p3, "pi_train": pi_master, "n_train": len(y_master)
    }
    print(f"  MCC(raw)={loo_p3_raw['mcc']:.4f} MCC(prior)={loo_p3_prior['mcc']:.4f}")
    print(f"  Boot-mean(prior)={loo_p3_prior['boot8020']['mean']:.4f} +/- {loo_p3_prior['boot8020']['std']:.4f}")
except Exception as e:
    print(f"  P3 HATA: {e}")
    all_results["P3_IsotonicCal"] = None

# ================================================================
print("\nP3: VennAbers (P0 + Venn-Abers calibration)...")
skf_va = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_master_proba = np.zeros(len(y_master))
for tri, vai in skf_va.split(X_master_le, y_master):
    m_fold = _lgbm_classifier()
    m_fold.fit(X_master_le.iloc[tri], y_master[tri])
    oof_master_proba[vai] = m_fold.predict_proba(X_master_le.iloc[vai])[:, 1]

va_cal = VennAbersCalibrator()
va_cal.fit(oof_master_proba.reshape(-1, 1), y_master)

p_p3_pah_raw = m_p0.predict_proba(X_pah_m_le)[:, 1]
p_p3_pah_cal = va_cal.predict_proba(p_p3_pah_raw.reshape(-1, 1))
if p_p3_pah_cal.ndim == 2:
    p_p3_pah_cal = p_p3_pah_cal[:, 1]

loo_p3_raw = loo_metrics(y_pah, p_p3_pah_cal, prior_shift=False)
loo_p3_prior = loo_metrics(y_pah, p_p3_pah_cal, prior_shift=True, pi_train=pi_master)

p_master_cal = va_cal.predict_proba(oof_master_proba.reshape(-1, 1))
if p_master_cal.ndim == 2:
    p_master_cal = p_master_cal[:, 1]
thr_p3 = select_threshold_8020_robust(y_master, p_master_cal)
train_p3 = train_metrics_at(y_master, p_master_cal, thr_p3)

all_results["P3_VennAbers"] = {
    "loo_raw": loo_p3_raw, "loo_prior": loo_p3_prior,
    "train": train_p3, "pi_train": pi_master, "n_train": len(y_master)
}
print(f"  MCC(raw)={loo_p3_raw['mcc']:.4f} MCC(prior)={loo_p3_prior['mcc']:.4f}")
print(f"  Boot-mean(prior)={loo_p3_prior['boot8020']['mean']:.4f} +/- {loo_p3_prior['boot8020']['std']:.4f}")

# ================================================================
# P4: COMBINED + BalancedBagging
# ================================================================
print("\nP4: COMBINED + BalancedBagging...")
bb_p4 = BalancedBaggingClassifier(
    estimator=_lgbm_classifier(),
    n_estimators=20,
    sampling_strategy="not minority",
    random_state=SEED,
    n_jobs=-1
)
bb_p4.fit(X_combined_le, y_combined)
p_p4_train = bb_p4.predict_proba(X_combined_le)[:, 1]
p_p4_pah = bb_p4.predict_proba(X_pah_c_le)[:, 1]

thr_p4 = select_threshold_8020_robust(y_combined, p_p4_train)
train_p4 = train_metrics_at(y_combined, p_p4_train, thr_p4)
loo_p4_raw = loo_metrics(y_pah, p_p4_pah, prior_shift=False)
loo_p4_prior = loo_metrics(y_pah, p_p4_pah, prior_shift=True, pi_train=pi_combined)

all_results["P4_COMBINED_BalBag"] = {
    "loo_raw": loo_p4_raw, "loo_prior": loo_p4_prior,
    "train": train_p4, "pi_train": pi_combined, "n_train": len(y_combined)
}
print(f"  MCC(raw)={loo_p4_raw['mcc']:.4f} MCC(prior)={loo_p4_prior['mcc']:.4f}")
print(f"  Boot-mean(prior)={loo_p4_prior['boot8020']['mean']:.4f} +/- {loo_p4_prior['boot8020']['std']:.4f}")

print("\n" + "="*70)
print("Tum stratejiler tamamlandi!")
print("="*70)

In [ ]:
# Cell 7: Sonuc Derleme

rows = []
for name, res in all_results.items():
    loo_p = res["loo_prior"]
    loo_r = res["loo_raw"]
    train = res["train"]
    boot = loo_p["boot8020"]
    rows.append({
        "Strateji": name,
        "n_train": res["n_train"],
        "LOO-MCC (raw)": round(loo_r["mcc"], 4),
        "LOO-MCC (prior)": round(loo_p["mcc"], 4),
        "LOO-F1": round(loo_p["f1"], 4),
        "LOO-AUC": round(loo_p["auc"], 4),
        "Precision": round(loo_p["precision"], 4),
        "Recall": round(loo_p["recall"], 4),
        "Boot-mean": round(boot["mean"], 4),
        "Boot-std": round(boot["std"], 4),
        "Boot-lo": round(boot["lo"], 4),
        "Boot-hi": round(boot["hi"], 4),
        "Train-F1": round(train["train_f1"], 4),
        "Train-MCC": round(train["train_mcc"], 4),
        "TN": loo_p["tn"], "FP": loo_p["fp"],
        "FN": loo_p["fn"], "TP": loo_p["tp"],
    })

results_df = pd.DataFrame(rows)
results_df = results_df.sort_values("LOO-MCC (prior)", ascending=False).reset_index(drop=True)

print("\n=== PAH Refinement Sonuclari (LOO-MCC prior siralama) ===\n")
display_cols = ["Strateji", "n_train", "LOO-MCC (raw)", "LOO-MCC (prior)", 
                "Boot-mean", "Boot-std", "Precision", "Recall", "TN", "FP", "FN", "TP", "Train-F1"]
print(results_df[display_cols].to_string(index=False))

# CSV kaydet
csv_path = os.path.join(RESULTS_DIR, "pah_results.csv")
results_df.to_csv(csv_path, index=False)
print(f"\nSonuclar kaydedildi: {csv_path}")

# NB16 referans
print("\n--- Referans: NB16 PAH en iyi = stack_lr %80/20 F1=0.515, CI=[0.35-0.57] ---")

In [ ]:
# Cell 8: Gorsellestirmeler

valid_res = {k: v for k, v in all_results.items() if v is not None}

# --- Fig 1: LOO-MCC karsilastirmasi (raw vs prior-shift) ---
fig, ax = plt.subplots(figsize=(10, 5))
names = list(valid_res.keys())
mcc_raw = [valid_res[n]["loo_raw"]["mcc"] for n in names]
mcc_prior = [valid_res[n]["loo_prior"]["mcc"] for n in names]
x = np.arange(len(names))
w = 0.35
bars1 = ax.bar(x - w/2, mcc_raw, w, label="MCC (raw)", color="steelblue", alpha=0.8)
bars2 = ax.bar(x + w/2, mcc_prior, w, label="MCC (prior-shift)", color="darkorange", alpha=0.8)
ax.set_ylabel("LOO-CV MCC")
ax.set_title("NB21 -- PAH: LOO-CV MCC Karsilastirmasi")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha="right", fontsize=8)
ax.legend()
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.3)
for b in bars1:
    ax.annotate(f"{b.get_height():.3f}", (b.get_x()+b.get_width()/2, b.get_height()), 
                ha="center", va="bottom", fontsize=7)
for b in bars2:
    ax.annotate(f"{b.get_height():.3f}", (b.get_x()+b.get_width()/2, b.get_height()), 
                ha="center", va="bottom", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig1_loo_mcc.png"), dpi=150)
plt.close()
print("fig1_loo_mcc.png kaydedildi")

# --- Fig 2: Boot-mean karsilastirmasi ---
fig, ax = plt.subplots(figsize=(10, 5))
boot_means = [valid_res[n]["loo_prior"]["boot8020"]["mean"] for n in names]
boot_stds = [valid_res[n]["loo_prior"]["boot8020"]["std"] for n in names]
colors = plt.cm.Set2(np.linspace(0, 1, len(names)))
bars = ax.bar(x, boot_means, yerr=boot_stds, capsize=5, color=colors, alpha=0.85)
ax.set_ylabel("Bootstrap %80/20 F1 (pathogenic)")
ax.set_title("NB21 -- PAH: Bootstrap %80/20 F1 (N=50, prior-shift)")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha="right", fontsize=8)
ax.axhline(y=0.515, color="red", linestyle="--", alpha=0.5, label="NB16 baseline (0.515)")
ax.legend()
for b in bars:
    ax.annotate(f"{b.get_height():.3f}", (b.get_x()+b.get_width()/2, b.get_height()), 
                ha="center", va="bottom", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig2_boot_mean.png"), dpi=150)
plt.close()
print("fig2_boot_mean.png kaydedildi")

# --- Fig 3: Confusion matrix (en iyi 3 strateji) ---
sorted_names = results_df["Strateji"].tolist()[:3]
fig, axes = plt.subplots(1, min(3, len(sorted_names)), figsize=(4.5*min(3, len(sorted_names)), 4))
if len(sorted_names) == 1:
    axes = [axes]
for i, name in enumerate(sorted_names):
    res = all_results[name]["loo_prior"]
    cm = np.array([[res["tn"], res["fp"]], [res["fn"], res["tp"]]])
    ax = axes[i]
    im = ax.imshow(cm, cmap="Blues", aspect="auto")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Benign", "Patho"])
    ax.set_yticklabels(["Benign", "Patho"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    boot = all_results[name]["loo_prior"]["boot8020"]
    ax.set_title(f"{name}\nMCC={res['mcc']:.3f}, Boot={boot['mean']:.3f}", fontsize=9)
    for ii in range(2):
        for jj in range(2):
            ax.text(jj, ii, str(cm[ii, jj]), ha="center", va="center", fontsize=14,
                   color="white" if cm[ii, jj] > cm.max()/2 else "black")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig3_confusion.png"), dpi=150)
plt.close()
print("fig3_confusion.png kaydedildi")

# --- Fig 4: Prior-shift posterior dagilimi (en iyi strateji) ---
best_name = sorted_names[0]
best_prob_raw = all_results[best_name]["loo_raw"]["prob"]
best_prob_adj = all_results[best_name]["loo_prior"]["prob"]
best_y = all_results[best_name]["loo_prior"]["y_true"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
for label, color, lbl in [(0, "green", "Benign"), (1, "blue", "Pathogenic")]:
    mask = best_y == label
    ax1.hist(best_prob_raw[mask], bins=20, alpha=0.5, color=color, label=f"{lbl} (n={mask.sum()})")
    ax2.hist(best_prob_adj[mask], bins=20, alpha=0.5, color=color, label=f"{lbl} (n={mask.sum()})")
ax1.axvline(all_results[best_name]["loo_raw"]["thr"], color="red", linestyle="--", 
            label=f"thr={all_results[best_name]['loo_raw']['thr']:.3f}")
ax2.axvline(all_results[best_name]["loo_prior"]["thr"], color="red", linestyle="--", 
            label=f"thr={all_results[best_name]['loo_prior']['thr']:.3f}")
ax1.set_title(f"{best_name} -- Raw Posterior")
ax1.legend(fontsize=8)
ax2.set_title(f"{best_name} -- Prior-Shift Adjusted")
ax2.legend(fontsize=8)
ax1.set_xlabel("P(pathogenic)")
ax2.set_xlabel("P(pathogenic)")
ax1.set_ylabel("Count")
ax2.set_ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig4_posterior.png"), dpi=150)
plt.close()
print("fig4_posterior.png kaydedildi")

In [ ]:
# Cell 9: Ozet & Tartisma

print("\n" + "="*80)
print("NB21 OZET & TARTISMA")
print("="*80)

# En iyi strateji
best = results_df.iloc[0]
print(f"\nEn iyi strateji: {best['Strateji']}")
print(f"  LOO-MCC (prior): {best['LOO-MCC (prior)']}")
print(f"  Boot-mean @%80/20: {best['Boot-mean']} +/- {best['Boot-std']}")
print(f"  Boot CI: [{best['Boot-lo']}, {best['Boot-hi']}]")
print(f"  Precision: {best['Precision']}, Recall: {best['Recall']}")
print(f"  Confusion: TN={best['TN']}, FP={best['FP']}, FN={best['FN']}, TP={best['TP']}")

print(f"\nNB16 referans: PAH stack_lr %80/20 F1=0.515")
delta = best['Boot-mean'] - 0.515
print(f"Fark: {delta:+.4f} (NB21 - NB16)")

print("\n--- Tum stratejiler (prior-shift MCC siralama) ---")
for _, row in results_df.iterrows():
    print(f"  {row['Strateji']:30s} MCC={row['LOO-MCC (prior)']:.4f}  Boot={row['Boot-mean']:.4f}+/-{row['Boot-std']:.4f}  P={row['Precision']:.3f} R={row['Recall']:.3f}  FP={row['FP']} FN={row['FN']}")

# Prior-shift etkisi
print("\n--- Prior-shift etkisi ---")
for name in all_results:
    raw = all_results[name]["loo_raw"]["mcc"]
    prior = all_results[name]["loo_prior"]["mcc"]
    delta_ps = prior - raw
    print(f"  {name:30s} raw={raw:.4f} -> prior={prior:.4f}  delta={delta_ps:+.4f}")

# Overfit kontrolu
print("\n--- Overfit kontrolu (train F1 vs LOO F1) ---")
for name in all_results:
    tf = all_results[name]["train"]["train_f1"]
    lf = all_results[name]["loo_prior"]["f1"]
    gap = tf - lf
    status = "OK" if gap < 0.15 else "ORTA" if gap < 0.25 else "YUKSEK"
    print(f"  {name:30s} train={tf:.4f} loo={lf:.4f} gap={gap:.4f} [{status}]")

In [ ]:
# Cell 10: Multi-seed dogrulama (en iyi 3 strateji)

print("Multi-seed 50/50 degerlendirme (en iyi 3 strateji)...")
top3 = results_df["Strateji"].tolist()[:3]

ms_results = {}
for name in top3:
    print(f"\n  {name}...")
    res = all_results[name]
    
    if "COMBINED" in name:
        X_train_le_temp = X_combined_le
        y_train_temp = y_combined
        X_panel_le_temp = X_pah_c_le
    else:
        X_train_le_temp = X_master_le
        y_train_temp = y_master
        X_panel_le_temp = X_pah_m_le
    
    if "BalancedBag" in name:
        def _fit_pred(X_tr, y_tr, X_te, y_te):
            bb = BalancedBaggingClassifier(
                estimator=_lgbm_classifier(), n_estimators=20,
                sampling_strategy="not minority", random_state=SEED, n_jobs=-1
            )
            bb.fit(X_train_le_temp, y_train_temp)
            return bb.predict_proba(X_te)[:, 1]
    elif "VennAbers" in name:
        def _fit_pred(X_tr, y_tr, X_te, y_te):
            m = _lgbm_classifier()
            m.fit(X_train_le_temp, y_train_temp)
            p_raw = m.predict_proba(X_te)[:, 1]
            p_cal = va_cal.predict_proba(p_raw.reshape(-1, 1))
            return p_cal[:, 1] if p_cal.ndim == 2 else p_cal
    else:
        def _fit_pred(X_tr, y_tr, X_te, y_te):
            m = _lgbm_classifier()
            m.fit(X_train_le_temp, y_train_temp)
            return m.predict_proba(X_te)[:, 1]
    
    ms = multiseed_eval(_fit_pred, X_panel_le_temp, y_pah)
    ms_results[name] = ms
    print(f"    Multi-seed F1: {ms['mean']:.4f} +/- {ms['std']:.4f}")

# Multi-seed sonuclari tabloya ekle
for name in ms_results:
    idx = results_df.index[results_df["Strateji"] == name].tolist()
    if idx:
        results_df.loc[idx[0], "MS-F1-mean"] = ms_results[name]["mean"]
        results_df.loc[idx[0], "MS-F1-std"] = ms_results[name]["std"]

# Guncellenmis CSV
results_df.to_csv(csv_path, index=False)
print(f"\nGuncellenmis sonuclar: {csv_path}")

In [ ]:
# Cell 11: PDF Rapor
from fpdf import FPDF

class PAHReport(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 10)
        self.cell(0, 8, "PAH Paneli Refinement Raporu | TEKNOFEST 2025", 0, 1, "C")
        self.set_draw_color(200, 50, 50)
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(4)
    
    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.cell(0, 10, f"Sayfa {self.page_no()}/{{nb}}", 0, 0, "C")
    
    def section(self, title):
        self.set_font("Helvetica", "B", 13)
        self.set_text_color(0, 102, 153)
        self.cell(0, 10, title, 0, 1, "L")
        self.set_text_color(0, 0, 0)
    
    def body(self, text):
        self.set_x(self.l_margin)
        self.set_font("Helvetica", "", 9)
        self.multi_cell(0, 5, text)
        self.set_x(self.l_margin)
        self.ln(2)
    
    def bullet(self, text):
        self.set_x(self.l_margin)
        self.set_font("Helvetica", "", 9)
        self.cell(5, 5, "-")
        self.multi_cell(0, 5, text)
        self.set_x(self.l_margin)
    
    def add_table(self, headers, data, col_widths=None):
        if col_widths is None:
            col_widths = [190 / len(headers)] * len(headers)
        self.set_font("Helvetica", "B", 8)
        self.set_fill_color(0, 102, 153)
        self.set_text_color(255, 255, 255)
        for i, h in enumerate(headers):
            self.cell(col_widths[i], 6, h, 1, 0, "C", True)
        self.ln()
        self.set_x(self.l_margin)
        self.set_text_color(0, 0, 0)
        self.set_font("Helvetica", "", 7)
        for row in data:
            for i, val in enumerate(row):
                self.cell(col_widths[i], 5, str(val), 1, 0, "C")
            self.ln()
            self.set_x(self.l_margin)
    
    def add_fig(self, path, w=170):
        if os.path.exists(path):
            self.image(path, x=(210-w)/2, w=w)
            self.ln(3)

pdf = PAHReport()
pdf.alias_nb_pages()
pdf.add_page()

# Baslik
pdf.set_font("Helvetica", "B", 18)
pdf.cell(0, 15, "PAH Paneli: Refinement Raporu", 0, 1, "C")
pdf.set_font("Helvetica", "", 10)
pdf.cell(0, 6, f"NB21 | {datetime.now().strftime('%Y-%m-%d')}", 0, 1, "C")
pdf.ln(5)

# Yonetici ozeti
pdf.section("Yonetici Ozeti")
best = results_df.iloc[0]
pdf.body(f"PAH panelinin performansini iyilestirmek icin 7 strateji karsilastirildi. "
         f"En iyi strateji: {best['Strateji']} (LOO-MCC={best['LOO-MCC (prior)']}, "
         f"Boot-mean={best['Boot-mean']}). NB16 referans: %80/20 F1=0.515.")

# Sonuc tablosu
pdf.section("1. Strateji Karsilastirmasi")
headers = ["Strateji", "n_train", "MCC(raw)", "MCC(prior)", "Boot-mean", "Boot-std", "Prec", "Recall", "FP", "FN"]
cw = [30, 15, 18, 18, 20, 16, 14, 14, 12, 12]
data = []
for _, row in results_df.iterrows():
    data.append([
        row["Strateji"][:25], int(row["n_train"]),
        f"{row['LOO-MCC (raw)']:.3f}", f"{row['LOO-MCC (prior)']:.3f}",
        f"{row['Boot-mean']:.3f}", f"{row['Boot-std']:.3f}",
        f"{row['Precision']:.3f}", f"{row['Recall']:.3f}",
        int(row["FP"]), int(row["FN"])
    ])
pdf.add_table(headers, data, cw)
pdf.ln(3)

# Figurler
pdf.section("2. LOO-MCC Karsilastirmasi")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig1_loo_mcc.png"))

pdf.add_page()
pdf.section("3. Bootstrap %80/20 F1")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig2_boot_mean.png"))

pdf.section("4. Confusion Matrix (En Iyi 3)")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig3_confusion.png"))

pdf.add_page()
pdf.section("5. Posterior Dagilimi (En Iyi Strateji)")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig4_posterior.png"))

# Tartisma
pdf.section("6. Tartisma")
pdf.body(f"PAH paneli (n=372, benign=62) icin 7 strateji test edildi. "
         f"n=62 benign CFTR'nin 3x kati -- bootstrap CI'lar istatistiksel olarak anlamli. "
         f"Prior-shift duzeltmesi (Saerens 2002) %83 patho egitim dagilimindan "
         f"%20 patho test dagiliminina adapte eder.")
pdf.body(f"NB16 referans: PAH stack_lr %80/20 F1=0.515. "
         f"En iyi strateji: {best['Strateji']} ({best['Boot-mean']:.3f}).")

# Kaydet
pdf_path = os.path.join(REPORTS_DIR_NB, "NB21_pah_refinement_report.pdf")
pdf.output(pdf_path)
print(f"PDF raporu olusturuldu: {pdf_path}")